[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rl-tools/hyperdrone/blob/main/docs/Overview.ipynb)

# HyperDrone Overview

The environment in five cells: three content-addressed ProcTHOR scenes, four drone models
flying in each, one simulation step, and every output modality the environment renders —
the shot of Figure 1 of the HyperDrone paper and its counterparts in two more scenes. The
paper's API appendix is generated from this notebook. Install `hyperdrone[examples]` to include the plotting dependencies. The first construction of an
environment configuration JIT-compiles it; later constructions load it from the cache.

In [ ]:
import importlib.util
import site
import sysconfig
from pathlib import Path

checkout = next((root for path in [Path.cwd(), *Path.cwd().parents]
                 for root in (path, path / "rl_tools")
                 if (root / "interface/python/hyperdrone/pyproject.toml").is_file()), None)
if checkout is not None:
    %pip install -q -e "{checkout}/interface/python/hyperdrone[examples]"
    site.addsitedir(sysconfig.get_paths()["purelib"])  # activates the editable install's .pth without a kernel restart
elif importlib.util.find_spec("hyperdrone") is None:  # fresh runtime (e.g. Colab)
    %pip install -q "hyperdrone[examples]"

The folded cell below holds only the false-color helpers of Figure 1 (depth, normals,
label palette, flow), the mosaic layout and a pose formatter.

In [ ]:
from textwrap import fill

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import hsv_to_rgb

import hyperdrone
from hyperdrone import render

hyperdrone.set_verbosity("warning")
%config InlineBackend.print_figure_kwargs = {"bbox_inches": "tight", "pad_inches": 0}
FLOW_MAX_PIXELS = 25.0


def pose(position, yaw_degrees):
    half = np.deg2rad(yaw_degrees) / 2
    return [*position, float(np.cos(half)), 0.0, 0.0, float(np.sin(half))]


def to_uint8(image):
    return image if image.dtype == np.uint8 else (np.clip(image, 0, 1) * 255).astype(np.uint8)


def depth_image(depth):
    finite = np.where(np.isfinite(depth), depth, np.nan)
    near = np.nanmin(finite, axis=(-2, -1), keepdims=True)  # per image
    far = np.nanmax(finite, axis=(-2, -1), keepdims=True)
    image = np.nan_to_num((far - finite) / (far - near) * 255).astype(np.uint8)
    return np.repeat(image[..., None], 3, axis=-1)


def normals_image(normals):
    valid = normals.any(axis=-1)
    image = np.zeros(normals.shape, dtype=np.uint8)
    image[valid] = np.rint((normals[valid] + 1) * 0.5 * 255)
    return image


def segmentation_image(labels, *, semantic=False):
    ids, inverse = np.unique(labels, return_inverse=True)
    palette = np.zeros((len(ids), 3), dtype=np.uint8)
    for index, label in enumerate(ids):
        if label == np.iinfo(np.uint32).max:
            continue  # a missed ray; instance ID zero is valid geometry
        if semantic and label == 0:
            palette[index] = 128  # undefined semantic class
        else:
            hue = (int(label) * 0.618033988749895 + 0.08) % 1.0
            palette[index] = np.rint(hsv_to_rgb((hue, 0.65, 0.95)) * 255)
    return palette[inverse].reshape(*labels.shape, 3)


def flow_image(flow, status):
    hue = np.mod(np.arctan2(flow[..., 1], flow[..., 0]) / (2 * np.pi), 1)
    saturation = np.clip(np.linalg.norm(flow, axis=-1) / FLOW_MAX_PIXELS, 0, 1)
    return hsv_to_rgb(np.stack([hue, saturation, status == render.FlowStatus.VALID], axis=-1))


def mosaic(rows, columns, dpi=200):
    columns = {title: to_uint8(tiles) for title, tiles in columns.items()}
    size = next(iter(columns.values())).shape[1]
    gap, band = size // 128, size // 4  # tile gap; label band above and left of the tiles
    step, fontsize = size + gap, band / 2.5 * 72 / dpi  # two lines of text per band
    canvas = np.full((band + len(rows) * step - gap, band + len(columns) * step - gap, 3), 255, dtype=np.uint8)
    figure = plt.figure(figsize=(canvas.shape[1] / dpi, canvas.shape[0] / dpi), dpi=dpi)
    axis = figure.add_axes([0, 0, 1, 1])
    for column, (title, tiles) in enumerate(columns.items()):
        x = band + column * step
        axis.text(x + size / 2, band / 2, fill(title, 13), ha="center", va="center", fontsize=fontsize)
        for row, tile in enumerate(tiles):
            y = band + row * step
            canvas[y:y + size, x:x + size] = tile
    for row, label in enumerate(rows):
        axis.text(band / 2, band + row * step + size / 2, fill(label, 13), rotation=90, ha="center", va="center", fontsize=fontsize)
    axis.imshow(canvas, interpolation="none")
    axis.axis("off")
    plt.show()

## Scene and drone table

Scenes are GLB files addressed by content hash: the `Train200` catalog pins the 200 ProcTHOR
training scenes, three of which are used here, and `conta` fetches each once into a shared
cache. Alternatively, you can also provide local filesystem paths instead. The drone table lists
the models that fly together in one instance, each a rigged GLB paired with the dynamics
model of the same name from the registry.

In [ ]:
import numpy as np
from hyperdrone import render
from hyperdrone.camera import CameraSpec, RGB, Depth, Segmentation, Normals, Flow, FlowStatus, ImageObservation
from hyperdrone import Config, HyperDrone
from hyperdrone.examples.data import rigged_model_path

catalog = render.datasets.Train200()
scenes = [catalog[f"ProcTHOR-Train-{index}"] for index in (1, 99, 72)]
MODELS = ("crazyflie_brushless", "arpl", "x500", "savagebee_pusher")
drones = [{"asset": rigged_model_path(name),
           "parameters": {"dynamics": {"dynamics": {"model": name}}}}
          for name in MODELS]

## Environment

`Config` is the compile-time part of an environment and its JIT key: resolution,
named camera profiles, typed outputs, motion-blur samples, agent layout and control/actuation
models. `HyperDrone` holds `num_worlds` worlds, each showing one scene to
`instances_per_world` instances. Here each scene gets one world with one instance of four
mutually visible agents; `env.total_instances` is therefore three. Agent 0 in each instance
carries the only camera, `front`, whose mount puts it at the pose of Figure 1.
`ImageObservation` selects this camera's float32 RGB for the policy. The two `Segmentation`
outputs label each pixel with its instance ID and its class ID; `FlowStatus` declares the
validity codes used to mask optical flow. Runtime camera overrides use the same `front` name.

In [ ]:
config = Config(preset="x500_fpv", num_agents=4, instances_per_world=1,
                num_worlds=len(scenes), width=1024, height=1024,
                control="motor", actuation="rotors",
                motion_blur_samples=16, dataset="procthor",
                cameras=(CameraSpec("front", outputs=(
                       RGB("float32"), Depth(), Segmentation(labels="instance"),
                       Segmentation(labels="semantic"), Normals(), Flow(), FlowStatus())),),
                   observation=ImageObservation(cameras=("front",)))
mount = {"position": [-0.2102, 0.2639, 0.3870],  # Figure 1's camera in
         "orientation": [0.6276, 0.2865, 0.2584, -0.6763]}  # agent 0 body frame
drones[0]["parameters"]["cameras"] = {"front": {"fov": 80, "mount": {"pose": mount}}}

## Initialization

Runtime parameters are a partial dict merged over the nominal values. Here the
initialization distribution is pinned to zero linear velocity with the propellers at full
speed and a random initial angular velocity of up to 1 rad/s per axis, which is what the
flow tiles will show.

In [ ]:
initial = {"guidance": 0, "max_angular_velocity": 1, "max_linear_velocity": 0.0,
           "min_rpm": 1, "max_rpm": 1}
parameters = {"dynamics": {"mdp": {"init": initial}}}
env = HyperDrone([scene.reference for scene in scenes], config=config,
                 drones=drones, seed=0, parameters=parameters)

## Reset and step

`reset` samples the scene placement and the poses. We pre-define spawn points for the drones such that they will be in the field of view of agent 0. `observe`
renders lazily, and `step` integrates 10 ms of dynamics for all agents. A policy would map
the observation to actions. With `control="motor"` and `actuation="rotors"`, actions are
normalized motor commands in `[-1, 1]`; here every agent applies full throttle (`+1`).

In [ ]:
formations = [  # (position, yaw in degrees) per agent
    [((-1.602, -3.994, 1.145), 65), ((-0.459, -4.226, 1.120), 30),
     ((-0.365, -5.578, 1.073), -35), ((-1.806, -4.461, 0.982), -20)],
    [((-13.976, -12.003, 1.103), 30), ((-13.768, -13.273, 0.667), 150),
     ((-14.423, -12.652, 0.718), -100), ((-13.466, -12.420, 0.642), 20)],
    [((-2.082, -13.649, 0.943), -139), ((-2.014, -11.854, 0.634), 45),
     ((-2.374, -12.688, 0.605), -150), ((-3.100, -13.230, 0.617), 80)]]
shots = [{"scene": scene.reference, "drone": [0, 1, 2, 3],
          "poses": [pose(position, yaw) for position, yaw in formation],
          "scene_translation": [0, 0, 0], "scene_yaw": [1, 0],
          "shutter_fraction": 0.2}
         for scene, formation in zip(scenes, formations)]
actions = np.ones((env.total_instances, env.action_dim), dtype=np.float32)
env.reset(spawn_points=shots)
env.observe()                               # first capture: the flow reference
env.step(actions)

## Outputs

`render()` acquires the pending capture after the step; reading an output does not render.
`env.outputs[descriptor].read()` returns an `(instances, selected_views, H, W[, C])`
snapshot and accepts `out=` buffers (including CUDA arrays on an OptiX environment).
Here every output selects the single `front` view, and each scene is one row of the mosaic.
`frames()` still returns the processed policy image, while the RGB output below contains
raw renderer pixels. `Flow()` is displacement to the previous capture, valid where the
`FlowStatus()` output equals `render.FlowStatus.VALID`.

In [ ]:
env.render()
mosaic([scene.id for scene in scenes], {
    "RGB": env.outputs[RGB("float32")].read()[:, 0],
    "Depth": depth_image(env.outputs[Depth()].read()[:, 0]),
    "Surface normals": normals_image(env.outputs[Normals()].read()[:, 0]),
    "Semantic segmentation": segmentation_image(
        env.outputs[Segmentation(labels="semantic")].read()[:, 0], semantic=True),
    "Instance segmentation": segmentation_image(
        env.outputs[Segmentation(labels="instance")].read()[:, 0]),
    "Optical flow": flow_image(
        env.outputs[Flow()].read()[:, 0], env.outputs[FlowStatus()].read()[:, 0]),
})